# Tiny Mitra: one token's journey
Run this notebook from the project directory. It imports the same Python files as training and Gradio. Read the prerequisite guide alongside it. Random weights illustrate calculations; they are not learned results.

In [ ]:
from pathlib import Path
import torch
from tokenizer import CharacterTokenizer
from config import ModelConfig
from model import TinyGPT
from data import render, batch_items
torch.set_num_threads(2)
torch.manual_seed(42)
tokenizer = CharacterTokenizer()
source = 'PATIENT John Lee TREATMENT DrugA'
record = render([('PATIENT','John Lee'),('TREATMENT','DrugA')])
record

## Text → tokens → IDs
The table address is an integer ID; it is not an embedding. Special markers and placeholders are atomic. New ASCII names retain their spelling.

In [ ]:
ids, offsets = tokenizer.encode_with_offsets(source)
list(zip(ids, [tokenizer.tokens[i] for i in ids], offsets))

## Token identity + position
An embedding table has one learned row per token. Position has a separate row per index. Addition preserves the feature width. These initial random vectors have no established clinical interpretation.

In [ ]:
model = TinyGPT(ModelConfig(len(tokenizer), d_model=16, d_ff=32, context=256))
x = torch.tensor([ids])
token_vectors = model.token_embedding(x)
position_vectors = model.position_embedding(torch.arange(x.shape[1]))
combined = token_vectors + position_vectors
print('IDs:', x.shape, 'token vectors:', token_vectors.shape, 'positions:', position_vectors.shape, 'combined:', combined.shape)
print('First token:', token_vectors[0,0].detach(), '\nPosition:', position_vectors[0].detach(), '\nSum:', combined[0,0].detach())

## Inside one head: Q, K, V and the causal mask
Q and K compare token features; V supplies the values to mix. B is batch size, T is token count, D is feature width, H is the number of heads and Dh=D/H. Scores have shape B,H,T,T. Each query row distributes probability over its present/past keys. These are random teaching weights, so no semantic role is inferred.

In [ ]:
import math
with torch.no_grad():
    block = model.blocks[0]
    attention = block.attention
    normalized = block.norm1(combined)
    B,T,D = normalized.shape
    H,Dh = attention.heads,attention.head_width
    def split(projection):
        return projection(normalized).view(B,T,H,Dh).transpose(1,2)
    Q,K,V = split(attention.q),split(attention.k),split(attention.v)
    scores = Q @ K.transpose(-2,-1) / math.sqrt(Dh)
    future = torch.ones(T,T,dtype=torch.bool).triu(1)
    weights = torch.softmax(scores.masked_fill(future,float('-inf')),dim=-1)
    mixed = weights @ V
    merged = mixed.transpose(1,2).contiguous().view(B,T,D)
    attention_update = attention.output(merged)
    assert torch.allclose(attention_update,attention(normalized))
    assert torch.allclose(weights.sum(-1),torch.ones(B,H,T))
    assert torch.count_nonzero(weights.masked_select(future[None,None])) == 0
    print('Q/K/V:',Q.shape,'scores:',scores.shape,'mixed:',mixed.shape)
    print('First query, head 0:',weights[0,0,0])
    print('Residual keeps width:',(combined+attention_update).shape)

## Finish a decoder block
The first residual adds the attention update to the input. A second normalization and FFN produce another update, followed by another residual addition. The next block repeats this process. Final normalization and the vocabulary projection produce one raw score per possible next token. Softmax converts scores to probabilities; choosing a token is a separate generation decision.

In [ ]:
with torch.no_grad():
    after_attention = combined + attention_update
    after_ffn = after_attention + block.ffn(block.norm2(after_attention))
    assert torch.allclose(after_ffn,block(combined,None))
    print('Block output:',after_ffn.shape)
    final_logits = model(x)
    next_probabilities = torch.softmax(final_logits[0,-1],dim=-1)
    print('Vocabulary scores:',final_logits.shape,'probability sum:',float(next_probabilities.sum()))
    print('Random-model candidate:',tokenizer.tokens[int(next_probabilities.argmax())])

## Training alignment
At `<OUTPUT>`, the next target is `P`. Before that, targets have ignore value -100. EOS remains scored. Padding is a separate mask.

In [ ]:
inputs, targets, valid = batch_items([record], tokenizer, model.config.context)
[(tokenizer.tokens[int(i)], None if int(y)==-100 else tokenizer.tokens[int(y)]) for i,y in zip(inputs[0],targets[0])]

## Forward → loss → gradient
Loss uses teacher forcing, so expected previous output tokens are present. This single backward pass calculates gradients; no optimizer update is made here.

In [ ]:
logits = model(inputs, valid, capture=True)
loss = torch.nn.functional.cross_entropy(logits.reshape(-1,len(tokenizer)), targets.reshape(-1), ignore_index=-100)
model.zero_grad()
loss.backward()
print('Logits:', logits.shape, 'loss:', float(loss), 'embedding gradient norm:', float(model.token_embedding.weight.grad.norm()))
attention = model.attention_matrices()[0]
print('Block 0 attention B,H,T,T:', attention.shape)
print('Final query, head 0:', attention[0,0,-1])

## Real checkpoint and free generation
This section uses trained weights only if present. A generated output can differ from the expected target. Inspect the actual result before interpreting it.

In [ ]:
from train import load_checkpoint
from generate import generate
checkpoint_path = Path('artifacts/checkpoints/transformer_2_seed42/best.pt')
if checkpoint_path.exists():
    trained, saved_tokenizer, metadata = load_checkpoint(checkpoint_path)
    print(metadata['run_id'], 'selected step', metadata['step'])
    print(generate(trained, saved_tokenizer, source))
else:
    print('No full-model checkpoint yet. Train or download one; do not substitute random weights.')

## Optional: the fixed-task-piece representation experiment
The first walkthrough used the character teaching model. The new experiment encodes declared schema/clinical pieces as single IDs and retains character spelling for other text. These are fixed pieces, not a vocabulary fitted to test names; a piece may also occur inside a longer word. The decoder, attention and gradient mathematics remain the same. Compare complete-note metrics across representations; per-token loss has different units.

In [ ]:
from task_tokenizer import TaskTokenizer
from task_runtime import load_task_checkpoint
task_tokenizer=TaskTokenizer()
for representation in (tokenizer,task_tokenizer):
    encoded=representation.encode(source)
    print(representation.metadata()['type'],len(encoded),[representation.tokens[i] for i in encoded])
task_checkpoint=Path('artifacts/mitra_final/checkpoints/transformer_2_seed42/latest.pt')
if task_checkpoint.exists():
    task_model,task_tokenizer,task_metadata=load_task_checkpoint(task_checkpoint)
    print('Actual saved update:',task_metadata['step'])
    print(generate(task_model,task_tokenizer,source))
else:
    print('Task-piece checkpoint not available yet; no result is invented.')

## My understanding
Write in your own words after reading and discussing: what does an ID select, what does position add, which targets contribute loss, and how does free generation differ from teacher forcing? We will discuss these one at a time. This notebook does not automatically mark learning complete.